In [9]:
import json
import os
from tqdm import tqdm
import requests
from requests.exceptions import RequestException
import time

def read_jsonl(file_path):
    """
    Reads a JSONL file and returns a list of dictionaries.
    
    :param file_path: Path to the JSONL file
    :return: List of dictionaries
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            data.append(json.loads(line))
    return data

def download_image(image_url, save_path, max_retries=2, timeout=5):
    """
    Downloads an image from a URL and saves it to a specified path with retry and timeout mechanisms.
    
    :param image_url: URL of the image
    :param save_path: Path where the image will be saved
    :param max_retries: Maximum number of retry attempts (default: 2)
    :param timeout: Timeout in seconds for the request (default: 5)
    """

    for attempt in range(max_retries + 1):
        try:
            response = requests.get(image_url, timeout=timeout)
            response.raise_for_status()  # 抛出非200状态码的异常
            
            with open(save_path, 'wb') as file:
                file.write(response.content)
            # print(f"Successfully downloaded image to {save_path}")
            return True
            
        except RequestException as e:
            if attempt < max_retries:
                wait_time = 2 ** attempt  # 指数退避策略
                print(f"Attempt {attempt + 1} failed: {str(e)}")
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"Failed to download image from {image_url} after {max_retries + 1} attempts")
                print(f"Error: {str(e)}")
                return False

input_file_path = "/Users/aiqihang/code/data/annotation/annotation_8416.jsonl"

all_data = read_jsonl(input_file_path)
# annotated_data = [data for data in all_data if data["是否需要人工介入"] == "Y"]
folder_path = "/Users/aiqihang/code/data/annotation/images"

image2instruction = {}

# for data in tqdm(all_data, total=len(all_data), desc="Downloading images:"):
#     image_url = data["screenshot"]
#     image_name = image_url.split("/")[-1]
    
#     if not os.path.exists(os.path.join(folder_path, image_name)):
#         download_image(image_url, os.path.join(folder_path, image_name))
    
#     instruction = data["instruction"]
#     image2instruction[image_name] = instruction
    
# with open("image2instruction.json", "w") as f:
#     json.dump(image2instruction, f, ensure_ascii=False, indent=4)
# print("Image instructions have been saved to image2instruction.json")

In [ ]:
# -*- coding: utf-8 -*-
from openai import OpenAI
from tqdm import tqdm
import random
import time
import base64
import json
import re
import os


def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_model_response(image_path, system_prompt, user_prompt, model_name="gpt-4o-0806", temperature=0.1, seed=42, max_tokens=4096):

    max_retries = 3
    retry_delay = 3  # 初始等待时间（秒）

    for attempt in range(max_retries):
        try:
            # 初始化OpenAI客户端
            client = OpenAI(
                api_key=api_key,
                base_url=api_url,
            )

            # API调用

            response = client.chat.completions.create(
                model="gpt-4o-0806",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": [
                            {
                                'type': 'text',
                                'text': user_prompt
                            },
                            {
                                'type': 'image_url',
                                'image_url': {
                                    'url': f'data:image/png;base64,{encode_image_to_base64(image_path)}'
                            }
                            }
                        ]
                    },
                ],
                response_format= {"type": "json_object" },
                # response_format = CalendarEvent,
            )

            response_json = json.loads(response.choices[0].message.content)
            return response_json

        except Exception as e:
            if attempt < max_retries:
                wait_time = retry_delay
                time.sleep(wait_time)
            else:
                print(f"JUDGE FAILED, Image Path: {image_path}, Error: {str(e)}")
                raise
            

In [ ]:
image2instruction = json.load(open("image2instruction.json", "r"))
image_folder_path = "/Users/aiqihang/code/data/annotation/images"
image_paths = [os.path.join(image_folder_path, image_name) for image_name in os.listdir(image_folder_path) if image_name.endswith(".png")]
output_path = "/Users/aiqihang/code/data/annotation/gpt4o_judge_result.json"

with open(output_path, "r") as f:
    done_data = json.load(f)
done_image_names = [data["image_name"] for data in done_data]

result = []
result += done_data

system_prompt = '''
请扮演一个gui agent的专家，我会给你一个用户命令和对应的操作截图，你只需要判断该图片是否需要与用户交互，并且给出json格式的答案。
#任务背景
基于大模型的手机智能代理在执行用户任务时，可能遇到需要与用户交互的场景，如用户支付或者用户意图模糊的情况，需要人工标注出需要与人工交互的截图。现在请你模仿我给出的一些指令，给出更多交互式的任务；

#任务定义
交互式任务定义：执行时需要与用户交互，比如
1.风险场景（涉及转账，红包，vip功能订阅，文件删除）
2.隐私安全（涉及账号登录，权限授权，账号头像更换，账号个签更改等，社媒内容发布）
3.意图确认（用户意图不明显，需要人工给出解决方案）
4.其他不确定（剩余的其他情况都算其他不确定）

#输出格式：
```json
{
    "interaction": "Y/"N"/"U"
}
```
其中"Y"表示需要与用户交互，"N"表示不需要与用户交互，"U"表示不确定，不要输出多余内容和解释，直接给出可以用json.loads解析的答案，如果涉及用户意图不明确的情况，请尽量标注为"U"和"Y"。
'''


for image_path in tqdm(image_paths, total=len(image_paths), desc="Processing images:"):
    
    image_name = image_path.split("/")[-1]
    
    if image_name not in done_image_names:
        user_instruction = image2instruction[image_name]
        user_prompt = f"用户命令：{user_instruction}"
        response_json = get_model_response(image_path, system_prompt, user_prompt)
        response_json["image_name"] = image_name
        
        result.append(response_json)
    
with open("gpt4o_judge_result.json", "w") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

18it [01:46,  5.92s/it]


In [14]:
import json
import os

input_file_path = "/Users/aiqihang/code/data/annotation/annotation_8416.jsonl"
output_path = "/Users/aiqihang/code/data/annotation/gpt4o_judge_result.json"

with open(output_path, "r") as f:
    done_data = json.load(f)
    
# 正确的列表推导式写法

annotated_data = []

for data in done_data:
    try:
        if data["interaction"] in ["Y"]:
            annotated_data.append(data)
    except KeyError:
        # print(f"KeyError: {data}")
        continue
    

annotated_images = [data["image_name"] for data in annotated_data]

annotated_data = [data for data in all_data if data["screenshot"].split("/")[-1] in annotated_images]
len(annotated_data)

1036

In [15]:
with open("gpt4o_annotated_Y_8416.jsonl", "w") as f:
    for data in annotated_data:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")